# Practica: Correlacion y causalidad (2.3)

**Asignatura:** Analisis y visualizacion de datos (GAD-2401)

## Objetivos
1. Simular una variable confusora y ver como genera correlacion "falsa".
2. Contrastar asociacion observada vs mecanismo real.
3. Practicar el lenguaje correcto al reportar hallazgos.

> Completa antes los ejercicios de `ejercicios/ejercicios-manuales.html` en el cuaderno.


## 1. Librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("OK")

## 2. Simulacion de confusora

- `temperatura` (confusora Z)
- `helado` depende de la temperatura
- `ataques` dependen de la temperatura (no del helado)

Medimos la correlacion entre helado y ataques.


In [ ]:
n = 200
temperatura = np.random.normal(28, 5, n)
helado = 10 + 2.5 * temperatura + np.random.normal(0, 5, n)
ataques = 1 + 0.15 * temperatura + np.random.normal(0, 0.8, n)
ataques = np.clip(ataques, 0, None)

df = pd.DataFrame({"temperatura": temperatura, "helado": helado, "ataques": ataques})

r_ha, p_ha = stats.pearsonr(df["helado"], df["ataques"])
r_th, _ = stats.pearsonr(df["temperatura"], df["helado"])
r_ta, _ = stats.pearsonr(df["temperatura"], df["ataques"])

print(f"helado vs ataques:      r = {r_ha:.3f}, p = {p_ha:.2e}")
print(f"temperatura vs helado:  r = {r_th:.3f}")
print(f"temperatura vs ataques: r = {r_ta:.3f}")
print("\nHay correlacion helado-ataques, pero el mecanismo corre por la temperatura.")

## 3. Visualizacion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.scatterplot(data=df, x="helado", y="ataques", ax=axes[0], alpha=0.7)
axes[0].set_title(f"Helado vs ataques\nr={r_ha:.2f}")
sns.scatterplot(data=df, x="temperatura", y="helado", ax=axes[1], alpha=0.7, color="C1")
axes[1].set_title(f"Temp vs helado\nr={r_th:.2f}")
sns.scatterplot(data=df, x="temperatura", y="ataques", ax=axes[2], alpha=0.7, color="C2")
axes[2].set_title(f"Temp vs ataques\nr={r_ta:.2f}")
plt.tight_layout()
plt.show()

## 4. Correlacion parcial (idea de "controlar" Z)

Correlacion entre residuos de helado|temp y ataques|temp.
Si la asociacion helado-ataques era solo por la confusora, esta correlacion parcial se acerca a 0.


In [ ]:
def residuos_vs(y, x):
    slope, intercept, *_ = stats.linregress(x, y)
    return y - (intercept + slope * x)

res_helado = residuos_vs(df["helado"].values, df["temperatura"].values)
res_ataques = residuos_vs(df["ataques"].values, df["temperatura"].values)
r_parcial, p_parcial = stats.pearsonr(res_helado, res_ataques)

print(f"Correlacion simple helado-ataques:  r = {r_ha:.3f}")
print(f"Correlacion parcial (control temp): r = {r_parcial:.3f}, p = {p_parcial:.2e}")
print("Al 'quitar' el efecto lineal de la temperatura, la asociacion se debilita mucho.")

## 5. Correlacion por ruido (muestras finitas)

In [ ]:
rs = []
for _ in range(500):
    a = np.random.normal(size=40)
    b = np.random.normal(size=40)
    rs.append(stats.pearsonr(a, b)[0])
rs = np.array(rs)
print(f"Media de r (ruido): {rs.mean():.3f}")
print(f"% de |r| > 0.3: {(np.abs(rs) > 0.3).mean()*100:.1f}%")
plt.hist(rs, bins=20, edgecolor="black", alpha=0.8)
plt.title("Distribucion de r entre dos ruidos independientes (n=40)")
plt.xlabel("Pearson r")
plt.tight_layout()
plt.show()

## 6. Ejercicio de redaccion

Con la simulacion helado/ataques, escribe:

1. Un **mal** titular periodistico.
2. Un **buen** parrafo de informe tecnico.


### Tu respuesta

1. Mal titular:

2. Buen parrafo:



## 7. Cierre

- Correlacion = asociacion; no prueba causa-efecto.
- Las confusoras pueden fabricar correlaciones fuertes.
- Experimentos, temporalidad y control acercan a la causalidad.
- El lenguaje del reporte debe ser cuidadoso.

**Fin de la practica 2.3 (cierre del bloque correlacion 2.1–2.3)**
